# JourneyGraph LLM Pipeline Demo

- Prerequisites: `.env` with `ANTHROPIC_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` set. Neo4j running.

---
## Setup

Follows logic of command-line run.py file:
1. `get_llm_config()` — hard-fails if `ANTHROPIC_API_KEY` is missing
2. `Neo4jManager()` — hard-fails if DB is unreachable
3. `SliceRegistry()` — validates YAML slices against live graph
4. `Planner()` — builds the LLM instance

SliceRegistry validation completes before LLM call to avoid wasting tokens

In [46]:
import os, sys, logging
os.chdir(os.path.expanduser("~/Downloads/DAMG 7374/journeygraph"))
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.common.config import get_llm_config
from src.common.logger import get_logger
from src.common.neo4j_tools import Neo4jManager
from src.llm.slice_registry import SliceRegistry
from src.llm.planner import Planner

log = get_logger(__name__)

llm_config = get_llm_config()
print(f"LLM config loaded — provider={llm_config.llm_provider}  "
      f"model={llm_config.llm_model}  max_tokens={llm_config.llm_max_tokens}")

db = Neo4jManager()
print("Neo4j connected")

registry = SliceRegistry(db, strict=False)
print(f"SliceRegistry loaded — domains: {registry.domains()}")

planner = Planner(registry, llm_config, strict=False)
print("Planner ready")

LLM config loaded — provider=anthropic  model=claude-haiku-4-5-20251001  max_tokens=512
Neo4j connected
SliceRegistry loaded — domains: ['accessibility', 'delay_propagation', 'transfer_impact']
Planner ready
SliceRegistry loaded — domains: ['accessibility', 'delay_propagation', 'transfer_impact']
Planner ready


### Sample Questions

In [47]:
USER_QUERY = "How many trips were cancelled on the Red Line yesterday?"

---
## Step 1: Planner - Domain Classififcation

The Planner is a deterministic Python classifier (no LLM call/I/O) that scores the query against domain vocabularies from the different KG layers and picks the top domain

In [48]:
stage1_result = planner.classify_only(USER_QUERY)

print("Planner domain scores:")
for domain, score in sorted(stage1_result.scores.items()):
    print(f"  {domain:<22} {score:.4f}")

print(f"\nTop domain: {stage1_result.domain!r}  "
      f"(score {stage1_result.scores[stage1_result.domain]:.4f})")

Planner domain scores:
  accessibility          0.0000
  delay_propagation      0.0000
  transfer_impact        1.0000

Top domain: 'transfer_impact'  (score 1.0000)


---
## Step 2: Planner - LLM Call

First LLM call decidses:
- **path** — `text2cypher` / `subgraph` / `both`
- **anchors** — stations, routes, dates, pathway nodes

JSON parse failure triggers one retry with a corrective nudge. On second failure, degrades to `text2cypher` with empty anchors and sets `parse_warning`.

In [49]:
planner_output = planner.run(USER_QUERY)

print(f"Query: {USER_QUERY!r}")
print("═" * 56)
print(f"  domain           : {planner_output.domain!r}")
print(f"  path             : {planner_output.path!r}")
print(f"  schema_slice_key : {planner_output.schema_slice_key!r}")
print(f"  rejected         : {planner_output.rejected}")
print(f"  rejection_message: {planner_output.rejection_message!r}")
print(f"  path_reasoning   : {planner_output.path_reasoning!r}")
print(f"  anchor_notes     : {planner_output.anchor_notes!r}")
print(f"  parse_warning    : {planner_output.parse_warning}")
print("  anchors:")
print(f"    stations       : {planner_output.anchors.stations}")
print(f"    routes         : {planner_output.anchors.routes}")
print(f"    dates          : {planner_output.anchors.dates}")
print(f"    pathway_nodes  : {planner_output.anchors.pathway_nodes}")

if planner_output.rejected:
    print("\nQuery rejected — pipeline stops here.")

2026-04-12 10:47:27,496 | INFO     | httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Query: 'How many trips were cancelled on the Red Line yesterday?'
════════════════════════════════════════════════════════
  domain           : 'transfer_impact'
  path             : 'text2cypher'
  schema_slice_key : 'transfer_impact'
  rejected         : False
  rejection_message: None
  path_reasoning   : 'This is a specific count query requiring a precise numeric answer about cancelled trips on a particular route for a given date.'
  anchor_notes     : 'Yesterday is a relative date that should be resolved to the actual previous calendar day from query execution time.'
  anchors:
    stations       : []
    routes         : ['Red Line']
    dates          : ['yesterday']
    pathway_nodes  : []


---
## Step 3: Anchor Resolution

Matches Planner's anchor strings to nodes in the graph using full-text index lookup
- Phase 1: candidate generation (full-text index, up to candidate_limit per mention)
- Phase 2: disambiguation (TopK by default, or TypeWeightedCoherenceStrategy)


In [50]:
from datetime import datetime, UTC
from src.llm.anchor_resolver import AnchorResolver

CANDIDATE_LIMIT = 1   # 1 = baseline (no disambiguation). >1 enables strategy.
STRATEGY = "topk"     # "topk" or "coherence"

disambiguation_strategy = None
if STRATEGY == "coherence":
    from src.llm.disambiguation_strategies import TypeWeightedCoherenceStrategy
    disambiguation_strategy = TypeWeightedCoherenceStrategy()

invocation_time = datetime.now(UTC)

resolver = AnchorResolver(
    db=db,
    invocation_time=invocation_time,
    strategy=disambiguation_strategy,
    candidate_limit=CANDIDATE_LIMIT,
)
resolutions = resolver.resolve(planner_output.anchors)

print(f"Resolver config: {resolver.config}")
print(f"Any resolved?   {resolutions.any_resolved}")
print(f"Resolved flat:  {resolutions.as_flat_dict()}")
if resolutions.failed:
    print(f"Failed anchors: {resolutions.failed}")

if not resolutions.any_resolved:
    print("\nZero anchors resolved. Please restate with a specific station, route, or date.\n")

2026-04-12 10:47:27,590 | INFO     | src.llm.anchor_resolver | anchor_resolver | k=1 baseline | disambiguation skipped | strategy=TopKStrategy
2026-04-12 10:47:27,590 | INFO     | src.llm.anchor_resolver | anchor_resolver | route resolved | 'Red Line' → ['RED']
2026-04-12 10:47:27,591 | INFO     | src.llm.anchor_resolver | anchor_resolver | date resolved | 'yesterday' → ['20260411']
2026-04-12 10:47:27,590 | INFO     | src.llm.anchor_resolver | anchor_resolver | route resolved | 'Red Line' → ['RED']
2026-04-12 10:47:27,591 | INFO     | src.llm.anchor_resolver | anchor_resolver | date resolved | 'yesterday' → ['20260411']


Resolver config: {'candidate_limit': 1, 'strategy': 'TopKStrategy'}
Any resolved?   True
Resolved flat:  {'Red Line': ['RED'], 'yesterday': ['20260411']}


---
## Step 4a: Subgraph Path - HopExpander and ContextSerializer

Runs only when planner_output.path set to "subgraph" or "both"

- HopExpander — bidirectional hop expansion from anchor nodes, constrained by `DomainExpansionConfig`
- ContextSerializer — serializes to a text block and enforces a 2 000-token budget (tiktoken `cl100k_base`)

In [51]:
from src.llm.subgraph_builder import SubgraphBuilder

subgraph_output = None

if planner_output.path in {"subgraph", "both"}:
    builder = SubgraphBuilder(db=db)
    subgraph_output = builder.run(
        planner_output,
        resolutions,
        resolver_config=resolver.config,
    )

    print("─── SubgraphOutput ──────────────────────────────────────")
    print(f"  domain            : {subgraph_output.domain!r}")
    print(f"  success           : {subgraph_output.success}")
    print(f"  failure_reason    : {subgraph_output.failure_reason!r}")
    print(f"  node_count        : {subgraph_output.node_count}")
    print(f"  trimmed           : {subgraph_output.trimmed}")
    print(f"  anchor_resolutions: {subgraph_output.anchor_resolutions}")
    print(f"  resolver_config   : {subgraph_output.resolver_config}")
    print(f"  provenance_nodes  : {len(subgraph_output.provenance_nodes)} node(s)")
    print("─── Subgraph Context Block ───────────────────────────────")
    print(subgraph_output.context if subgraph_output.context else "  (empty)")
else:
    print(f"Subgraph path not selected, defaulting to {planner_output.path!r}")

Subgraph path not selected, defaulting to 'text2cypher'


---
## Step 4b: Text2Cypher QueryWriter

Runs only when planner_output.path set to "text2cypher" or "both".

QueryWriter:
1. Loads conventions.json and .cypher files for the domain
2. Builds a system prompt with conventions and example Cypher patterns
3. Makes a single LLM call to generate a Cypher query
4. Parses the response into cypher_query and provides chain-of-thought comments

In [52]:
from src.llm.query_writer import run_query_writer

query_writer_output = None

if planner_output.path in {"text2cypher", "both"}:
    query_writer_output = run_query_writer(USER_QUERY, planner_output)

    print("[Query Writer Output]")
    print("\nCypher Query:")
    print(query_writer_output.cypher_query)
    print("\nReasoning:")
    print(query_writer_output.cot_comments)
else:
    print(f"Text2Cypher path not selected, defaulting to{planner_output.path!r}).")

2026-04-12 10:47:31,961 | INFO     | httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


[Query Writer Output]

Cypher Query:
MATCH (r:Route {route_short_name: 'Red Line'})
       -[:HAS_TRIP]->(t:Trip)
WHERE t.is_cancelled = true
  AND t.service_date = date('yesterday')
RETURN COUNT(t) AS cancelled_trip_count

Reasoning:
I can see you're querying for cancelled trips on the Red Line yesterday, but there's a **schema mismatch** that prevents me from answering this query.

## The Issue

Your query requires the **`schedule`** schema (which contains trips, routes, and service calendars), but you've specified `transfer_impact` as the target schema.

The system conventions and examples you've provided describe the **`physical`** schema layer:
- Station/Platform/Pathway nodes and relationships
- Physical infrastructure (elevators, escalators, fare gates)
- Connectivity patterns

## What You Need

To answer "How many trips were cancelled on the Red Line yesterday?" I would need:

1. **Correct schema**: `schedule` (not `transfer_impact`)
2. **Data nodes**: 
   - `Trip` nodes with c

---
## Step 5: Cypher Validation

Validates the generated Cypher against the live Neo4j graph and the graph schema:

1. Syntax check via Neo4j EXPLAIN
2. Checks that every node/relationship label in the query appears in the slice's property_registry
5. If all checks pass, query is run and results returned

In [53]:
from src.llm.cypher_validator import validate_and_log_cypher

if query_writer_output:
    schema_slice = registry.get(planner_output.schema_slice_key)

    result = validate_and_log_cypher(
        query_writer_output.cypher_query,
        schema_slice,
        schema_slice.property_registry,
        db.driver,
        log,
    )

    print(f"Valid:   {result.valid}")
    print(f"Errors:  {result.errors}")
    if result.valid:
        print(f"Results: {result.results}")
else:
    print("No Cypher query was generated — nothing to validate.")

2026-04-12 10:47:32,211 | WARNING  | neo4j.notifications | Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `is_cancelled` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=9, offset=93>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 93, 'line': 3, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "EXPLAIN MATCH (r:Route {route_short_name: 'Red Line'})\n       -[:HAS_TRIP]->(t:Trip)\nWHERE t.is_cancelled = true\n  AND t.service_date = date('yesterday')\nRETURN COUNT(t) AS cancelled_trip_count"
2026-04-12 10:47:32,215 | WARNING  | neo4j.notificatio

Valid:   False
Errors:  ['Label \'HAS_TRIP\' not in whitelist for schema slice \'SchemaSlice(domain=\'transfer_impact\', nodes=[\':Interruption:Cancellation\', \':Trip\', \':Route\', \':Route:Rail\', \':Route:Bus\', \':Station\', \':Platform\', \':Date\'], relationships=[RelationshipTriple(from_label=\'Route\', rel_type=\'SERVES\', to_label=\'Station\'), RelationshipTriple(from_label=\'Date\', rel_type=\'ON_DATE\', to_label=\'Interruption\'), RelationshipTriple(from_label=\'Interruption\', rel_type=\'AFFECTS_TRIP\', to_label=\'Trip\'), RelationshipTriple(from_label=\'Interruption\', rel_type=\'AFFECTS_ROUTE\', to_label=\'Route\'), RelationshipTriple(from_label=\'Interruption\', rel_type=\'AFFECTS_STOP\', to_label=\'Station\'), RelationshipTriple(from_label=\'Trip\', rel_type=\'SCHEDULED_AT\', to_label=\'Platform\'), RelationshipTriple(from_label=\'Trip\', rel_type=\'FOLLOWS\', to_label=\'RoutePattern\'), RelationshipTriple(from_label=\'RoutePattern\', rel_type=\'BELONGS_TO\', to_label=

In [54]:
db.close()